In [1]:
import os
from google.colab import drive

# Mount a disk
gdrive_dir = '/content/drive/'
drive.mount(gdrive_dir, force_remount=True)

# Look for two folders
folders = [
    '/content/drive/MyDrive/!corruption/',
    '/content/drive/MyDrive/!records/',
    '/content/drive/MyDrive/!transcripts/',
    '/content/drive/MyDrive/!keys/'
]

# Check them
for folder in folders:
    print(f"\n=== Files in: {folder} ===")
    for dirname, _, filenames in os.walk(folder):
        for filename in filenames:
            print(os.path.join(dirname, filename))

Mounted at /content/drive/

=== Files in: /content/drive/MyDrive/!corruption/ ===
/content/drive/MyDrive/!corruption/youtube_downloader_cyr_merged.xlsx
/content/drive/MyDrive/!corruption/youtube_data.xlsx
/content/drive/MyDrive/!corruption/channel_ai_analysis.xlsx
/content/drive/MyDrive/!corruption/channel_ai_processing_log.xlsx
/content/drive/MyDrive/!corruption/playlist_join_diagnostics.xlsx
/content/drive/MyDrive/!corruption/playlist_items_diagnostics.xlsx
/content/drive/MyDrive/!corruption/playlist_ai_analysis.xlsx
/content/drive/MyDrive/!corruption/playlist_ai_processing_log.xlsx
/content/drive/MyDrive/!corruption/video_selection_diagnostics.xlsx
/content/drive/MyDrive/!corruption/video_ai_processing_log.xlsx
/content/drive/MyDrive/!corruption/video_ai_analysis.xlsx
/content/drive/MyDrive/!corruption/video_selection_diagnostics_no_shorts.xlsx
/content/drive/MyDrive/!corruption/video_ai_processing_log_no_shorts.xlsx
/content/drive/MyDrive/!corruption/video_ai_analysis_no_shorts.xls

In [ ]:
pip install openpyxl

In [ ]:
pip install openai

In [ ]:
pip install -q yt-dlp

In [ ]:
apt-get update -qq

In [ ]:
apt-get install -y ffmpeg

In [ ]:
from pathlib import Path
from typing import Optional, Dict, Any, List
import json
import time
import pandas as pd
from openai import OpenAI

# =========================
# CONFIGURATION
# =========================

INPUT_XLSX = Path("/content/drive/MyDrive/!corruption/Final_dataset_with_transcripts.xlsx")
OUTPUT_XLSX = Path("/content/drive/MyDrive/!corruption/Final_transcript_ai_analysis.xlsx")
LOG_XLSX = Path("/content/drive/MyDrive/!corruption/transcript_ai_processing_log.xlsx")
API_KEY_PATH = Path("/content/drive/MyDrive/!keys/openai.txt")

SHEET_NAME = "dataset_with_transcripts"
MODEL_NAME = "gpt-5.1"

VIDEO_ID_COL = "video_id"
TRANSCRIPT_COL = "transcript_text"

MAX_RETRIES = 3
REQUEST_TIMEOUT = 180
SLEEP_BETWEEN_REQUESTS = 1.0
MAX_COMPLETION_TOKENS = 2500

SKIP_EXISTING = True
LIMIT_ROWS: Optional[int] = None

MAX_TRANSCRIPT_CHARS = 25000


SYSTEM_INSTRUCTION = """
You are analyzing YouTube video transcripts for a sociological research project.

Research topic:
"Social practices of public audit and corruption exposure in Ukraine."

Analyze only the provided transcript and metadata.
Do not use external knowledge.
Do not invent facts.

Return ONLY valid JSON.
Do not use markdown.
Do not include long verbatim transcript fragments.
Each string value must be concise.
No string value may exceed 300 characters.
In key_evidence_markers, write short paraphrased evidence markers, not long direct quotes.

Use this JSON schema:

{
  "main_topic": "string",
  "corruption_domain": "string",
  "public_audit_practices": ["string"],
  "audit_mechanisms": ["string"],
  "evidence_sources": ["string"],
  "actors_accused_or_checked": ["string"],
  "institutions_mentioned": ["string"],
  "documents_or_data_used": ["string"],
  "fieldwork_elements": ["string"],
  "whistleblower_or_insider_role": "string",
  "law_enforcement_role": "string",
  "journalistic_role": "string",
  "audience_role": "string",
  "narrative_type": "string",
  "emotional_tone": "string",
  "call_to_action": "string",
  "public_accountability_model": "string",
  "corruption_relevance_from_transcript": 0.0,
  "public_audit_relevance_from_transcript": 0.0,
  "metadata_score_consistency": "string",
  "key_evidence_markers": ["string"],
  "short_summary": "string",
  "analytical_note": "string"
}

Definitions:

corruption_domain:
Examples: public procurement, forestry, military, healthcare, judiciary, local government, political elites, business ownership, customs, public finance, unknown.

public_audit_practices:
Identify concrete practices such as:
- journalistic investigation
- open data analysis
- document analysis
- declaration analysis
- procurement monitoring
- field inspection
- interview with insider
- request to public institution
- appeal to law enforcement
- comparison of official and factual data
- exposure of hidden ownership
- monitoring of public officials
- public naming and shaming

narrative_type:
Examples:
- investigative exposure
- analytical commentary
- political discussion
- institutional critique
- field report
- interview-based audit
- mixed

metadata_score_consistency:
Assess whether the transcript confirms the metadata-based relevance score:
- confirms_high_relevance
- partially_confirms
- contradicts_or_weakens
- unclear

Give concise but analytically meaningful answers.
""".strip()


def load_api_key() -> str:
    key = API_KEY_PATH.read_text(encoding="utf-8").strip()
    if not key:
        raise ValueError("API key file is empty.")
    return key


def safe_str(value) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    return str(value).strip()


def truncate_text(text: str, max_chars: int) -> str:
    text = safe_str(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + "\n\n[TRUNCATED]"


def build_user_prompt(row: pd.Series) -> str:
    data = {
        "video_id": safe_str(row.get("video_id", "")),
        "channel_title": safe_str(row.get("channel_title", "")),
        "video_title": safe_str(row.get("video_title", "")),
        "published_at": safe_str(row.get("published_at", "")),
        "duration_seconds": safe_str(row.get("duration_seconds", "")),
        "api_section": safe_str(row.get("api_section", "")),
        "playlist_titles": safe_str(row.get("playlist_titles", "")),
        "channel_relevance_score": safe_str(row.get("channel_relevance_score", "")),
        "playlist_max_relevance_score": safe_str(row.get("playlist_max_relevance_score", "")),
        "video_relevance_score": safe_str(row.get("video_relevance_score", "")),
        "inclusion_status": safe_str(row.get("inclusion_status", "")),
        "corruption_relevance_category": safe_str(row.get("corruption_relevance_category", "")),
        "public_audit_relevance_category": safe_str(row.get("public_audit_relevance_category", "")),
        "transcript_text": truncate_text(
            safe_str(row.get(TRANSCRIPT_COL, "")),
            MAX_TRANSCRIPT_CHARS
        )
    }

    return (
        "Analyze the following YouTube transcript and metadata for the research topic:\n"
        "\"Social practices of public audit and corruption exposure in Ukraine.\"\n\n"
        "DATA JSON:\n"
        f"{json.dumps(data, ensure_ascii=False, indent=2)}"
    )


def parse_json_response(text: str) -> Dict[str, Any]:
    text = text.strip()

    if text.startswith("```"):
        text = text.strip("`").strip()
        if text.lower().startswith("json"):
            text = text[4:].strip()

    return json.loads(text)


def call_model(client: OpenAI, row: pd.Series) -> Dict[str, Any]:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_INSTRUCTION},
            {"role": "user", "content": build_user_prompt(row)}
        ],
        timeout=REQUEST_TIMEOUT,
        max_completion_tokens=MAX_COMPLETION_TOKENS,
        response_format={"type": "json_object"}
    )

    content = response.choices[0].message.content

    if not content or not content.strip():
        raise ValueError("Model returned empty response.")

    return parse_json_response(content)


def load_existing_results() -> pd.DataFrame:
    if not OUTPUT_XLSX.exists():
        return pd.DataFrame()

    try:
        return pd.read_excel(OUTPUT_XLSX, sheet_name="transcript_ai_analysis")
    except Exception:
        return pd.DataFrame()


def existing_keys(existing_df: pd.DataFrame) -> set:
    if existing_df.empty or "video_id" not in existing_df.columns:
        return set()
    return set(existing_df["video_id"].dropna().astype(str).str.strip())


def list_to_pipe(value) -> str:
    if isinstance(value, list):
        return " | ".join(str(x) for x in value)
    return safe_str(value)


def save_outputs(results: List[Dict[str, Any]], log_rows: List[Dict[str, Any]]) -> None:
    with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
        pd.DataFrame(results).to_excel(
            writer,
            index=False,
            sheet_name="transcript_ai_analysis"
        )

        pd.DataFrame(log_rows).to_excel(
            writer,
            index=False,
            sheet_name="processing_log"
        )

    pd.DataFrame(log_rows).to_excel(LOG_XLSX, index=False)


def main() -> None:
    api_key = load_api_key()
    client = OpenAI(api_key=api_key)

    df = pd.read_excel(INPUT_XLSX, sheet_name=SHEET_NAME)

    if VIDEO_ID_COL not in df.columns:
        raise ValueError(f"Missing column: {VIDEO_ID_COL}")

    if TRANSCRIPT_COL not in df.columns:
        raise ValueError(f"Missing column: {TRANSCRIPT_COL}")

    df[VIDEO_ID_COL] = df[VIDEO_ID_COL].astype(str).str.strip()
    df[TRANSCRIPT_COL] = df[TRANSCRIPT_COL].fillna("").astype(str)

    df = df[df[TRANSCRIPT_COL].str.strip().ne("")].copy()

    if LIMIT_ROWS is not None:
        df = df.head(LIMIT_ROWS).copy()

    existing_df = load_existing_results()
    processed_ids = existing_keys(existing_df) if SKIP_EXISTING else set()

    results: List[Dict[str, Any]] = []
    log_rows: List[Dict[str, Any]] = []

    if not existing_df.empty and SKIP_EXISTING:
        results.extend(existing_df.to_dict(orient="records"))

    print(f"Loaded rows with transcripts: {len(df)}")
    print(f"Already processed: {len(processed_ids)}")

    for idx, row in df.iterrows():
        video_id = safe_str(row.get(VIDEO_ID_COL, ""))

        if not video_id:
            log_rows.append({
                "row_index": idx,
                "video_id": "",
                "status": "failed",
                "error": "Missing video_id"
            })
            continue

        if SKIP_EXISTING and video_id in processed_ids:
            print(f"Skipping existing: {video_id}")
            continue

        print(f"Processing {idx}: {video_id} / {safe_str(row.get('video_title', ''))[:80]}")

        last_error = None

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                ai = call_model(client, row)

                result_row = {
                    "video_id": video_id,
                    "channel_title": safe_str(row.get("channel_title", "")),
                    "video_title": safe_str(row.get("video_title", "")),
                    "published_at": safe_str(row.get("published_at", "")),
                    "duration_seconds": safe_str(row.get("duration_seconds", "")),
                    "api_section": safe_str(row.get("api_section", "")),
                    "playlist_titles": safe_str(row.get("playlist_titles", "")),
                    "video_relevance_score": safe_str(row.get("video_relevance_score", "")),
                    "channel_relevance_score": safe_str(row.get("channel_relevance_score", "")),
                    "playlist_max_relevance_score": safe_str(row.get("playlist_max_relevance_score", "")),

                    "main_topic": ai.get("main_topic", ""),
                    "corruption_domain": ai.get("corruption_domain", ""),
                    "public_audit_practices": list_to_pipe(ai.get("public_audit_practices", [])),
                    "audit_mechanisms": list_to_pipe(ai.get("audit_mechanisms", [])),
                    "evidence_sources": list_to_pipe(ai.get("evidence_sources", [])),
                    "actors_accused_or_checked": list_to_pipe(ai.get("actors_accused_or_checked", [])),
                    "institutions_mentioned": list_to_pipe(ai.get("institutions_mentioned", [])),
                    "documents_or_data_used": list_to_pipe(ai.get("documents_or_data_used", [])),
                    "fieldwork_elements": list_to_pipe(ai.get("fieldwork_elements", [])),
                    "whistleblower_or_insider_role": ai.get("whistleblower_or_insider_role", ""),
                    "law_enforcement_role": ai.get("law_enforcement_role", ""),
                    "journalistic_role": ai.get("journalistic_role", ""),
                    "audience_role": ai.get("audience_role", ""),
                    "narrative_type": ai.get("narrative_type", ""),
                    "emotional_tone": ai.get("emotional_tone", ""),
                    "call_to_action": ai.get("call_to_action", ""),
                    "public_accountability_model": ai.get("public_accountability_model", ""),
                    "corruption_relevance_from_transcript": ai.get("corruption_relevance_from_transcript", ""),
                    "public_audit_relevance_from_transcript": ai.get("public_audit_relevance_from_transcript", ""),
                    "metadata_score_consistency": ai.get("metadata_score_consistency", ""),
                    "key_evidence_markers": list_to_pipe(ai.get("key_evidence_markers", [])),
                    "short_summary": ai.get("short_summary", ""),
                    "analytical_note": ai.get("analytical_note", "")
                }

                results.append(result_row)

                log_rows.append({
                    "row_index": idx,
                    "video_id": video_id,
                    "status": "ok",
                    "error": ""
                })

                last_error = None
                break

            except Exception as exc:
                last_error = exc
                print(f"  Attempt {attempt} failed: {exc}")

                if attempt < MAX_RETRIES:
                    time.sleep(attempt * 3)

        if last_error is not None:
            log_rows.append({
                "row_index": idx,
                "video_id": video_id,
                "status": "failed",
                "error": str(last_error)
            })

        save_outputs(results, log_rows)
        time.sleep(SLEEP_BETWEEN_REQUESTS)

    print("Done.")
    print(f"Saved to: {OUTPUT_XLSX}")
    print(f"Log saved to: {LOG_XLSX}")


if __name__ == "__main__":
    main()

Loaded rows with transcripts: 194
Already processed: 5
Skipping existing: RH5_pQl-5sQ
Skipping existing: mkp-CuYAyvQ
Skipping existing: rgXf4vyXYd8
Skipping existing: g5WVovcErQo
Skipping existing: 04tcNfl9jXs
Processing 5: XrkP4QWoHJM / «Я тренировал ГУР — а меня везут в бусике за уклонение». Антон Черный о беспреде
Processing 6: FETaAE7c_lk / 200 квадратів у Москві: як живе родина пасинка Медведчука
Processing 7: N2ewcDz7NUs / Бойовики спалились ЗНОВ 😎Ми знайшли базу «Пятнашки» – військові В#ЇБАЛИ
Processing 8: R0TJj4zIkGc / Розвод з технікою: як не потрапити на фейкові гаджети?
Processing 9: wWSV9A8JOms / Від «Кварталу» до «Мідаса»: офшори і родинний люкс експомічника Зеленського
Processing 10: Ba1d9iZ0wEE / 10 миллиардов в никуда: как Украина кормит чиновников оккупированных областей. О
Processing 11: RqqciDP5vEo / Романенко: почему "простая" украинская элита от Кучмы до Зеленского работает по 
Processing 12: xSK1qoeueik / "Волшебники все свалили": Романенко о ликвидации Второй Укр